In [32]:
import json
import logging
import os
from pathlib import Path

import boto3
import ipynbname
import pandas as pd
from dotenv import load_dotenv

In [33]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [34]:
p = Path(ipynbname.path()).parent.parent

format = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
filename_path = p / 'logs' / 'transform_data.log'
filename_path.parent.mkdir(parents=True, exist_ok=True)
logging.basicConfig(format=format, filename=filename_path, encoding='utf-8', level=logging.INFO)
logger = logging.getLogger(__name__)

load_dotenv(dotenv_path=p / '.env')

def extract_data():
    if not os.getenv('EXTRACT_DATETIME'):
        logger.error("EXTRACT_DATETIME is not set in the environment variables.")
        raise TypeError("Expected string for EXTRACT_DATETIME, received None (EXTRACT_DATETIME is not set in the environment variables).")
    
    if not os.getenv('S3_BUCKET_NAME'):
            logger.error("S3_BUCKET_NAME is not set in the environment variables.")
            raise TypeError("Expected string for S3_BUCKET_NAME, received None (S3_BUCKET_NAME is not set in the environment variables).")
    
    if not os.getenv('AWS_ACCESS_KEY_ID'):
        logger.error("AWS_ACCESS_KEY_ID is not set in the environment variables.")
        raise TypeError("Expected string for AWS_ACCESS_KEY_ID, received None (AWS_ACCESS_KEY_ID is not set in the environment variables).")

    if not os.getenv('AWS_SECRET_ACCESS_KEY'):
        logger.error("AWS_SECRET_ACCESS_KEY is not set in the environment variables.")
        raise TypeError("Expected string for AWS_SECRET_ACCESS_KEY, received None (AWS_SECRET_ACCESS_KEY is not set in the environment variables).")
    

    try:
        s3_client = boto3.client(
            's3', 
            aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'), 
            aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
        )
        response = s3_client.get_object(
            Bucket=os.getenv('S3_BUCKET_NAME'), 
            Key=f"bronze/mercadolivre/iphone-18-pro-max/{os.getenv('EXTRACT_DATETIME')}/bronze_data.json"
        )
        data = json.loads(response['Body'].read().decode('utf-8'))
        logger.info("bronze_data.json obtained from Amazon S3")
        return data
    except Exception as e:
        logger.error(f"Error occurred while extracting data from Amazon S3: {e}")
        raise

def create_dataframe() -> pd.DataFrame:
    data = extract_data()
    df = pd.json_normalize(data)
    return df

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    df_data_items = df.explode('data.items', ignore_index=True)
    df_items = pd.json_normalize(df_data_items['data.items'])
    df = pd.concat(
        [df_data_items, df_items],
        axis=1
    )
    return df

df = create_dataframe()
df = normalize_df(df)
df.head()


,requestId,executionId,data.source,data.type,data.url,data.requestUrl,data.extractedAt,data.query,data.totalResults,data.primaryResults,data.page,data.resultsPerPage,data.offset,data.nextPage,data.nextPageUrl,data.items,url,imageUrl,sku,categoryId,domainId,name,condition,currency,currencyRaw,price,highlight,isBestSeller,isPowerSeller,powerSellerStatusTitle,ean,sellerId,sellerName,sellerCity,sellerState,sellerCountry,aggregateRating.rating,aggregateRating.reviewCount,aggregateRating
0,79aa76a6-138f-492e-a3a5-074857a44c62,a7e35580-6215-48da-84ba-ace304235140,mercadolivre.com.br,plp,https://lista.mercadolivre.com.br/iphone-18-pro-max,https://lista.mercadolivre.com.br/iphone-18-pro-max,2026-11-01T14:01:17.950Z,iphone 18 pro max,50,50,1,48,0,2.0,https://lista.mercadolivre.com.br/celulares-telefones/celulares-smartphones/iphone/iphone-18-pro-max/iphone-18-pro-max_Desde_49_NoIndex_True,"{'url': 'https://www.mercadolivre.com.br/iphone-18-pro-max-256gb-preto-distribuidor-autorizado/p/MLB1077855502', 'imageUrl': 'https://http2.mlstatic.com/D_Q_NP_2X_672207-MLA116010620562_092026-E.webp', 'sku': 'MLB5214725561', 'categoryId': 'MLB1055', 'domainId': 'MLB-CELLPHONES', 'name': 'iPhone 18 Pro Max 256GB Preto - Distribuidor Autorizado', 'condition': None, 'currency': 'BRL', 'currencyRaw': 'R$', 'price': 11699, 'highlight': 'LOJA OFICIAL APPLE', 'isBestSeller': False, 'isPowerSeller': False, 'powerSellerStatusTitle': None, 'aggregateRating': {'rating': 4.9, 'reviewCount': None}, 'ean': None, 'sellerId': None, 'sellerName': 'Apple', 'sellerCity': None, 'sellerState': None, 'sellerCountry': None}",https://www.mercadolivre.com.br/iphone-18-pro-max-256gb-preto-distribuidor-autorizado/p/MLB1077855502,https://http2.mlstatic.com/D_Q_NP_2X_672207-MLA116010620562_092026-E.webp,MLB5214725561,MLB1055,MLB-CELLPHONES,iPhone 18 Pro Max 256GB Preto - Distribuidor Autorizado,NaN,BRL,R$,11699,LOJA OFICIAL APPLE,False,False,None,None,None,Apple,None,None,None,4.9,NaN,NaN
1,79aa76a6-138f-492e-a3a5-074857a44c62,a7e35580-6215-48da-84ba-ace304235140,mercadolivre.com.br,plp,https://lista.mercadolivre.com.br/iphone-18-pro-max,https://lista.mercadolivre.com.br/iphone-18-pro-max,2026-11-01T14:01:17.950Z,iphone 18 pro max,50,50,1,48,0,2.0,https://lista.mercadolivre.com.br/celulares-telefones/celulares-smartphones/iphone/iphone-18-pro-max/iphone-18-pro-max_Desde_49_NoIndex_True,"{'url': 'https://www.mercadolivre.com.br/iphone-18-pro-max-1tb-glacial-distribuidor-autorizado/p/MLB1077855641', 'imageUrl': 'https://http2.mlstatic.com/D_Q_NP_2X_849153-MLA116010390632_092026-E.webp', 'sku': 'MLB5214721245', 'categoryId': 'MLB1055', 'domainId': 'MLB-CELLPHONES', 'name': 'iPhone 18 Pro Max 1TB Glacial - Distribuidor Autorizado', 'condition': None, 'currency': 'BRL', 'currencyRaw': 'R$', 'price': 15749, 'highlight': 'LOJA OFICIAL APPLE', 'isBestSeller': False, 'isPowerSeller': False, 'powerSellerStatusTitle': None, 'aggregateRating': {'rating': 4.9, 'reviewCount': None}, 'ean': None, 'sellerId': None, 'sellerName': 'Apple', 'sellerCity': None, 'sellerState': None, 'sellerCountry': None}",https://www.mercadolivre.com.br/iphone-18-pro-max-1tb-glacial-distribuidor-autorizado/p/MLB1077855641,https://http2.mlstatic.com/D_Q_NP_2X_849153-MLA116010390632_092026-E.webp,MLB5214721245,MLB1055,MLB-CELLPHONES,iPhone 18 Pro Max 1TB Glacial - Distribuidor Autorizado,NaN,BRL,R$,15749,LOJA OFICIAL APPLE,False,False,None,None,None,Apple,None,None,None,4.9,NaN,NaN
2,79aa76a6-138f-492e-a3a5-074857a44c62,a7e35580-6215-48da-84ba-ace304235140,mercadolivre.com.br,plp,https://lista.mercadolivre.com.br/iphone-18-pro-max,https://lista.mercadolivre.com.br/iphone-18-pro-max,2026-11-01T14:01:17.950Z,iphone 18 pro max,50,50,1,48,0,2.0,https://lista.mercadolivre.com.br/celulares-telefones/celulares-smartphones/iphone/iphone-18-pro-max/iphone-18-pro-max_Desde_49_NoIndex_True,"{'url': 'https://www.mercadolivre.com.br/iphone-18-pro-max-512gb-preto-distribuidor-autorizado/p/MLB1077855516', 'imageUrl': 'https://http2

In [35]:
drop_columns = [
    'requestId', 'executionId', 'data.source', 'data.type', 'data.url', 
    'data.requestUrl', 'data.query', 'data.totalResults', 'data.primaryResults', 
    'data.page', 'data.resultsPerPage', 'data.offset', 'data.nextPage', 
    'data.nextPageUrl', 'data.items', 'aggregateRating'
]

def drop_unnecessary(df: pd.DataFrame, drop_columns: list) -> pd.DataFrame:
    df = df.drop(drop_columns, axis=1)
    return df

df = drop_unnecessary(df, drop_columns)
resume = pd.DataFrame({
    'dtype': df.dtypes,
    'total': len(df),
    'not_null': df.notna().sum(),
    'nulls': df.isna().sum(),
    'unique': df.nunique()
})
print(resume)

                               dtype  total  not_null  nulls  unique
data.extractedAt                 str     50        50      0       2
url                              str     50        50      0      48
imageUrl                         str     50        50      0      42
sku                              str     50        50      0      48
categoryId                       str     50        49      1       1
domainId                         str     50        50      0       1
name                             str     50        50      0      36
condition                        str     50        26     24       2
currency                         str     50        50      0       1
currencyRaw                      str     50        50      0       1
price                          int64     50        50      0      40
highlight                        str     50         8     42       4
isBestSeller                    bool     50        50      0       1
isPowerSeller                   bo

In [36]:
columns_to_rename = {
    'url': 'url',
    'sku': 'sku',
    'data.extractedAt': 'extracted_at',
    'imageUrl': 'image_url',
    'categoryId': 'category_id',
    'domainId': 'domain_id',
    'name': 'name',
    'condition': 'condition',
    'currency': 'currency',
    'currencyRaw': 'currency_raw',
    'price': 'price',
    'highlight': 'highlight',
    'isBestSeller': 'is_best_seller',
    'isPowerSeller': 'is_power_seller',
    'powerSellerStatusTitle': 'power_seller_status_title',
    'ean': 'ean',
    'sellerId': 'seller_id',
    'sellerName': 'seller_name',
    'sellerCity': 'seller_city',
    'sellerState': 'seller_state',
    'sellerCountry': 'seller_country',
    'aggregateRating.rating': 'rating',
    'aggregateRating.reviewCount': 'review_count',
}

def rename_columns(df: pd.DataFrame, columns_to_rename: dict) -> pd.DataFrame:
    df = df.rename(columns=columns_to_rename)

    return df

df = rename_columns(df, columns_to_rename)

In [37]:
qtd_unica = df[['currency', 'currency_raw']].drop_duplicates().shape[0]
print(f"Combinações únicas: {qtd_unica}")

Combinações únicas: 1


In [38]:
qtd_unica = df[['sku', 'ean', 'image_url', 'category_id', 'domain_id', 'name', 'condition']].drop_duplicates().shape[0]
print(f"Combinações únicas: {qtd_unica}")

Combinações únicas: 48


In [39]:
qtd_unica = df[['highlight']].drop_duplicates().shape[0]
print(f"Combinações únicas: {qtd_unica}")

Combinações únicas: 5


In [40]:
qtd_unica = df[['seller_id', 'seller_name', 'seller_city', 'seller_state', 'seller_country', 'is_best_seller', 'is_power_seller', 'power_seller_status_title']].drop_duplicates().shape[0]
print(f"Combinações únicas: {qtd_unica}")

Combinações únicas: 3


In [41]:
qtd_unica = df[['url']].drop_duplicates().shape[0]
print(f"Combinações únicas: {qtd_unica}")

Combinações únicas: 48


###########

In [42]:
def normalize_format(df: pd.DataFrame) -> pd.DataFrame:
    #extracted_at
    df['extracted_at'] = pd.to_datetime(df["extracted_at"])
    df['extracted_at'] = df["extracted_at"].dt.strftime('%Y-%m-%dT%H-%M-%S')

    #name
    df['name'] = (df['name']
                  .str.strip()
                  .str.replace(r"\s+", " ", regex=True)
                  .str.replace(r"(\d+)\s*gb", r"\1GB", case=False, regex=True)
                  .str.replace(r"(\d+)\s*tb", r"\1TB", case=False, regex=True)
                  )

    return df

df = normalize_format(df)

In [43]:
for x, y in df.items():
    print(f"{y}\n\n")
    print(f"{x}\n\n")

0     2026-11-01T14-01-17
1     2026-11-01T14-01-17
2     2026-11-01T14-01-17
3     2026-11-01T14-01-17
4     2026-11-01T14-01-17
5     2026-11-01T14-01-17
6     2026-11-01T14-01-17
7     2026-11-01T14-01-17
8     2026-11-01T14-01-17
9     2026-11-01T14-01-17
10    2026-11-01T14-01-17
11    2026-11-01T14-01-17
12    2026-11-01T14-01-17
13    2026-11-01T14-01-17
14    2026-11-01T14-01-17
15    2026-11-01T14-01-17
16    2026-11-01T14-01-17
17    2026-11-01T14-01-17
18    2026-11-01T14-01-17
19    2026-11-01T14-01-17
20    2026-11-01T14-01-17
21    2026-11-01T14-01-17
22    2026-11-01T14-01-17
23    2026-11-01T14-01-17
24    2026-11-01T14-01-17
25    2026-11-01T14-01-17
26    2026-11-01T14-01-17
27    2026-11-01T14-01-17
28    2026-11-01T14-01-17
29    2026-11-01T14-01-17
30    2026-11-01T14-01-17
31    2026-11-01T14-01-17
32    2026-11-01T14-01-17
33    2026-11-01T14-01-17
34    2026-11-01T14-01-17
35    2026-11-01T14-01-17
36    2026-11-01T14-01-17
37    2026-11-01T14-01-17
38    2026-1